# **Who**: David

## **Task**: find out what phrases/keywords are important for the positively annotated class with respect to the negatives.

Ingredients:
* simply word frequencies: first remove stopwords and extract the word counts (use sklearn's CountVectorizer perhaps), maybe do this per section of the text
* use Jacopo's models in combination with a tool like LIME or SHAP.
* aggregate the embeddings of the medical terms in the texts 

In [1]:
import pandas as pd

import string

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
# Necessary resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

In [2]:
df_consults = pd.read_parquet(r"T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240826\consult_texts.parquet")

df_afsprak = pd.read_sas(
    r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240822\consult_20240822.sas7bdat',
        encoding='latin1')

df_afsprak.studyId_0831 = df_afsprak.studyId_0831.astype('uint64')

df_afsprak = df_afsprak.drop([
    'Consult_id', 'ConsultStatus', 'type_code_original',
    'type_display_original','created', 'title', 'author_usercode',
    'initial_author_usercode', 'custodian_Organization_value',
    'SpecialismeNaam', 'initial_custodian_Organization_v',
    'SpecialismeNaamInitial', 'LocationCode', 'LocationName',
    'PatientProtected', 'section_title_code_original',
    'section_title_display_original', 'section_text',
    'section_text_contentType', 'orderedBy', 'index_date'
    ],
    axis='columns')

In [3]:
df_afsprak = df_afsprak.groupby('studyId_0831').last()

df_merged = pd.merge(df_consults, df_afsprak, 
    right_index=True, left_on='studyId_0831', how='left')

df_merged['delta'] = (df_merged.date_x - df_merged.date_y).dt.days

df_merged_g = df_merged.groupby('studyId_0831').last()
df_merged_g = df_merged_g.query('delta >= -365 and delta <= 0')

In [ ]:
df_merged_g['text'].str.split().str.len().sort_values(ascending=False)

### Stopwords removal from the consults dataset

In [5]:
# Carica le stopwords per la lingua olandese
stop_words = stopwords.words('dutch')

# Funzione per rimuovere stopwords e punteggiatura
def remove_stopwords(text):
    # Rimuovere punteggiatura
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Rimuovere stopwords e convertire in minuscolo
    text = ' '.join([word for word in text.lower().split() if word not in stop_words])
    
    return text

# Applica la funzione al testo
df_merged_g['clean_text'] = df_merged_g['text'].apply(remove_stopwords)

# Contare le parole in ciascun testo
df_merged_g['word_count'] = df_merged_g['clean_text'].str.split().str.len()

# Ordinare il DataFrame in base al conteggio delle parole
df_merged_g = df_merged_g.sort_values(by='word_count')

In [ ]:
df_merged_g['clean_text'].str.split().str.len().sort_values(ascending=False)

In [ ]:
Id = 47686465397
print(df_merged_g.loc[Id, 'text'].split())

print(df_merged_g.loc[Id, 'clean_text'].split())

print(df_merged_g.loc[Id, 'clean_text'])

print(df_merged_g.loc[Id, 'text'])

### 

In [ ]:
# Crea un'istanza di CountVectorizer
vectorizer = CountVectorizer(lowercase=True)

# Trasforma la colonna 'text' in una matrice di conteggio
X = vectorizer.fit_transform(df_consults['clean_text'])

# Ottieni i nomi delle caratteristiche (parole)
feature_names = vectorizer.get_feature_names_out()

# Somma i conteggi per ottenere le parole più comuni
word_counts = X.toarray().sum(axis=0)

# Mostra le parole più comuni
print(word_counts)

# Crea un DataFrame con i conteggi delle parole
word_count_df = pd.DataFrame({'word': feature_names, 'count': word_counts})

# Ordina il DataFrame in base al conteggio delle parole
word_count_df = word_count_df.sort_values(by='count', ascending=False)

# Mostra le parole più comuni
print(word_count_df)